# Decision Tree Model:-

> Here is the complete BigQuery ML (BQML) code to train a Decision Tree Classifier on the famous Titanic.csv dataset to predict passenger survival (Survived = 0 or 1).

# Important Concepts for Tree Models in BQML:-
1. **Model Type**: In BigQuery, single Decision Trees and ensembles are implemented using RANDOM_FOREST_CLASSIFIER (with num_parallel_tree = 1 to create a single Decision Tree) or BOOSTED_TREE_CLASSIFIER (XGBoost).
2. **Feature Importance**: Linear and Logistic models use ML.WEIGHTS, but Tree models do not have linear weights. Instead, we use ML.FEATURE_IMPORTANCE to see which features split the tree the most (e.g., Sex, Fare, Pclass).
3. **Missing Data in Titanic**:
    - Age has missing values → Filled with the median.
    - Embarked has missing values  → Filled with the mode ('S').

# Complete Code:-

```sql
-- Create a schema/dataset in the US region
CREATE SCHEMA IF NOT EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox`
OPTIONS(
  location = 'US'
);


-- Cleanse and prepare data for Tree training
CREATE OR REPLACE TABLE `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_titanic` AS
WITH stats AS (
  SELECT
    APPROX_QUANTILES(Age, 2)[OFFSET(1)] AS median_age,
    APPROX_QUANTILES(Fare, 2)[OFFSET(1)] AS median_fare
  FROM `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_titanic`
  WHERE Age IS NOT NULL AND Fare IS NOT NULL
)
SELECT
  -- Cast Survived to INT64 or STRING as the classification target
  CAST(Survived AS INT64) AS Survived,
  CAST(Pclass AS INT64) AS Pclass,
  CAST(Sex AS STRING) AS Sex,
  -- Impute missing Age with median
  CAST(COALESCE(Age, (SELECT median_age FROM stats)) AS FLOAT64) AS Age,
  CAST(SibSp AS INT64) AS SibSp,
  CAST(Parch AS INT64) AS Parch,
  -- Impute missing Fare with median
  CAST(COALESCE(Fare, (SELECT median_fare FROM stats)) AS FLOAT64) AS Fare,
  -- Impute missing Embarked with 'S' (most frequent port)
  CAST(COALESCE(Embarked, 'S') AS STRING) AS Embarked
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_titanic`;



-- Train a Decision Tree Classifier
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`
OPTIONS (
  model_type = 'RANDOM_FOREST_CLASSIFIER',
  input_label_cols = ['Survived'],
  
  -- Set exactly to 10 trees
  num_parallel_tree = 10,
  max_tree_depth = 6,
  
  -- 80% train / 20% test split
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2,
  auto_class_weights = TRUE
) AS
SELECT
  Survived,
  Pclass,
  Sex,
  Age,
  SibSp,
  Parch,
  Fare,
  Embarked
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_titanic`;



-- Evaluate the Decision Tree performance
SELECT
  *
FROM
  ML.EVALUATE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`);



-- Check False Positives and False Negatives
SELECT
  *
FROM
  ML.CONFUSION_MATRIX(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`);



-- Check which features are most important in the tree splits
SELECT
  feature,
  importance_weight,
  importance_gain
FROM
  ML.FEATURE_IMPORTANCE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`)
ORDER BY
  importance_gain DESC;


-- Predict survival for hypothetical new passengers
SELECT
  predicted_Survived,
  predicted_Survived_probs,
  Pclass,
  Sex,
  Age,
  Fare
FROM
  ML.PREDICT(
    MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`,
    (
      -- Passenger 1: 1st class, Female, solo traveler
      SELECT 
        CAST(1 AS INT64) AS Pclass,
        CAST('female' AS STRING) AS Sex,
        CAST(29.0 AS FLOAT64) AS Age,
        CAST(0 AS INT64) AS SibSp,
        CAST(0 AS INT64) AS Parch,
        CAST(211.3 AS FLOAT64) AS Fare,
        CAST('S' AS STRING) AS Embarked
      UNION ALL
      -- Passenger 2: 3rd class, Male, traveling with family
      SELECT 
        CAST(3 AS INT64) AS Pclass,
        CAST('male' AS STRING) AS Sex,
        CAST(22.0 AS FLOAT64) AS Age,
        CAST(1 AS INT64) AS SibSp,
        CAST(1 AS INT64) AS Parch,
        CAST(7.25 AS FLOAT64) AS Fare,
        CAST('S' AS STRING) AS Embarked
    )
  );




-- Clean up all created resources
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_titanic`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_titanic`;
DROP SCHEMA IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox` CASCADE;
```

# Complete Step by Step Code for Decision Tree Model

- **Step 1**: Create the Schema (Dataset)
```sql
-- Create a schema/dataset in the US region
CREATE SCHEMA IF NOT EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox`
OPTIONS(
  location = 'US'
);
```

---
```bash
gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_titanic
```


- **Step 2**: Data Cleaning & Preprocessing -> We drop identifiers like PassengerId, Name, and Ticket, and impute missing values for Age and Embarked.

```sql
-- Cleanse and prepare data for Tree training
CREATE OR REPLACE TABLE `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_titanic` AS
WITH stats AS (
  SELECT
    APPROX_QUANTILES(Age, 2)[OFFSET(1)] AS median_age,
    APPROX_QUANTILES(Fare, 2)[OFFSET(1)] AS median_fare
  FROM `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_titanic`
  WHERE Age IS NOT NULL AND Fare IS NOT NULL
)
SELECT
  -- Cast Survived to INT64 or STRING as the classification target
  CAST(Survived AS INT64) AS Survived,
  CAST(Pclass AS INT64) AS Pclass,
  CAST(Sex AS STRING) AS Sex,
  -- Impute missing Age with median
  CAST(COALESCE(Age, (SELECT median_age FROM stats)) AS FLOAT64) AS Age,
  CAST(SibSp AS INT64) AS SibSp,
  CAST(Parch AS INT64) AS Parch,
  -- Impute missing Fare with median
  CAST(COALESCE(Fare, (SELECT median_fare FROM stats)) AS FLOAT64) AS Fare,
  -- Impute missing Embarked with 'S' (most frequent port)
  CAST(COALESCE(Embarked, 'S') AS STRING) AS Embarked
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_titanic`;
```

---

- **Step 3**: Standard Random Forest (Recommended)
If you want to train an ensemble forest (usually 50–100 trees for optimal accuracy):

```sql
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`
OPTIONS (
  model_type = 'RANDOM_FOREST_CLASSIFIER',
  input_label_cols = ['Survived'],
  
  -- Set exactly to 10 trees
  num_parallel_tree = 10,
  max_tree_depth = 6,
  
  -- 80% train / 20% test split
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2,
  auto_class_weights = TRUE
) AS
SELECT
  Survived,
  Pclass,
  Sex,
  Age,
  SibSp,
  Parch,
  Fare,
  Embarked
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_titanic`;
```

---

- **Step 4**: Evaluate Model Performance -> This outputs standard classification metrics: Accuracy, Precision, Recall, F1-Score, and ROC AUC.

```sql
-- Evaluate the Decision Tree performance
SELECT
  *
FROM
  ML.EVALUATE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`);
```

#  Option 2: Train Exactly 1 Single Decision Tree:- 
- If your assignment requires strictly 1 decision tree, use BOOSTED_TREE_CLASSIFIER with max_iterations = 1:
```sql
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['Survived'],
  
  max_iterations = 1,         -- Trains exactly 1 single decision tree
  max_tree_depth = 6,
  
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2,
  auto_class_weights = TRUE
) AS
SELECT
  Survived,
  Pclass,
  Sex,
  Age,
  SibSp,
  Parch,
  Fare,
  Embarked
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_titanic`;
```
---


- **Step 5**: Confusion Matrix -> See how many survivors vs. non-survivors were predicted correctly or misclassified.

```sql
-- Check False Positives and False Negatives
SELECT
  *
FROM
  ML.CONFUSION_MATRIX(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`);
```

---


- **Step 6**: Feature Importance (Which features matter most?) -> Instead of weights, tree models score how much each feature contributes to splitting the data.

```sql
-- Check which features are most important in the tree splits
SELECT
  feature,
  importance_weight,
  importance_gain
FROM
  ML.FEATURE_IMPORTANCE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`)
ORDER BY
  importance_gain DESC;
```

---

- **Step 7**: Make Predictions on Unseen Passengers -> We provide exact typed test passengers to predict survival:
    - **Passenger 1**: 1st Class Female, Age 29, high fare (High survival probability expected).
    - **Passenger 2**: 3rd Class Male, Age 22, low fare (Low survival probability expected).

```sql
-- Predict survival for hypothetical new passengers
SELECT
  predicted_Survived,
  predicted_Survived_probs,
  Pclass,
  Sex,
  Age,
  Fare
FROM
  ML.PREDICT(
    MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`,
    (
      -- Passenger 1: 1st class, Female, solo traveler
      SELECT 
        CAST(1 AS INT64) AS Pclass,
        CAST('female' AS STRING) AS Sex,
        CAST(29.0 AS FLOAT64) AS Age,
        CAST(0 AS INT64) AS SibSp,
        CAST(0 AS INT64) AS Parch,
        CAST(211.3 AS FLOAT64) AS Fare,
        CAST('S' AS STRING) AS Embarked
      UNION ALL
      -- Passenger 2: 3rd class, Male, traveling with family
      SELECT 
        CAST(3 AS INT64) AS Pclass,
        CAST('male' AS STRING) AS Sex,
        CAST(22.0 AS FLOAT64) AS Age,
        CAST(1 AS INT64) AS SibSp,
        CAST(1 AS INT64) AS Parch,
        CAST(7.25 AS FLOAT64) AS Fare,
        CAST('S' AS STRING) AS Embarked
    )
  );
```

---

- **Step 8**: Clean Up Resources (Optional)

```sql
-- Clean up all created resources
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.titanic_tree_model`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_titanic`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_titanic`;
DROP SCHEMA IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox` CASCADE;
```